In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
curr_experiment = 'more_cnn_layers'

In [ ]:
import os

In [ ]:
checkpoints = "/content/drive/MyDrive/dl_ocr_checkpoints"
os.makedirs(checkpoints, exist_ok=True)

In [ ]:
checkpoint_path = f"{checkpoints}/{curr_experiment}_best_model.pth"

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from collections import OrderedDict

class CRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # cnn to extract features from the images
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),nn.MaxPool2d((2, 1)),
            nn.Conv2d(512, 512, 2), nn.BatchNorm2d(512), nn.ReLU(),
        )

        self.height_pool = nn.AdaptiveAvgPool2d((1, None))

        # use Bi-LSTM as the RNN to train recognition
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        # input to linear layer is twice the hidden size since using bidirectional LSTM
        self.fc = nn.Linear(256*2, vocab_size)

    def forward(self, x):
        features = self.cnn(x) # (batch_size, C, H, W)

        # collapse height to interpret it as a sequence over time
        features = self.height_pool(features) # (batch_size, C, 1, W)
        features = features.squeeze(2) # (batch_size, C, W)

        # reorder so they are formatted for RNN
        features = features.permute(0,2,1) # (W, batch_size, C)

        seq, _ = self.rnn(features)
        out = self.fc(seq)

        return out

In [ ]:
def collate_fn(batch):
    images, labels, texts = zip(*batch)

    # right pad image widths to max width
    max_w = max(img.shape[2] for img in images)
    padded = []
    for img in images:
        pad_w = max_w - img.shape[2]
        padded.append(torch.nn.functional.pad(img, (0, pad_w), value=-1.0)) # -1.0 is black
    images_tensor = torch.stack(padded)  # (batch_size, 1, 32, W_max)

    # CTC loss requires labels as a single flat 1D tensor
    labels_concat = torch.cat(labels)

    # CTC also needs to know where each label ends since they're all concatenated together
    label_lens = torch.tensor([len(l) for l in labels], dtype=torch.long)

    # CTC needs this to know the sequence length for each sample
    input_lens = torch.tensor([img.shape[2] // 4 - 1 for img in images], dtype=torch.long)

    return images_tensor, labels_concat, label_lens, input_lens, texts

In [ ]:
from PIL import Image
class ResizeKeepAspect:
    def __init__(self, height: int):
        self.height = height

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size

        # scale width proportionally to preserve aspect ratio
        new_w = max(1, int(w * self.height / h))  # max(1) guards against rounding to 0
        return img.resize((new_w, self.height), Image.BILINEAR)

In [ ]:
from torchvision import transforms
def get_transforms(img_height: int = 32, augment: bool = False):
    base = [
        ResizeKeepAspect(img_height), # scale width
        transforms.ToTensor(), # PIL -> tensor, scales to [0, 1]
        transforms.Normalize(mean=[0.5], std=[0.5]),  # re-center to [-1, 1]
    ]
    if augment:
        # training pipeline adds warp and blur to simulate real world handwriting photos
        base = [
            ResizeKeepAspect(img_height),
            transforms.RandomAffine(degrees=2, translate=(0.02, 0.02)),  # slight tilt/shift
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),    # simulate blurry images
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ]
    return transforms.Compose(base)

In [ ]:
CHARS = " !\"'()*+,-./:;?abcdefghijklmnopqrstuvwxyz0123456789"
class Vocab:
    def __init__(self, chars: str = CHARS):
        # blank token at index 0 (required by CTC)
        self.blank_idx = 0
        # shift all real characters up by 1 to leave index 0 free for blank
        self.char2idx = {ch: i + 1 for i, ch in enumerate(chars)}
        self.idx2char = {i + 1: ch for i, ch in enumerate(chars)}
        self.size = len(chars) + 1

    def encode(self, text: str) -> list[int]:
        text = text.lower()
        # skip any characters not in vocab
        return [self.char2idx[ch] for ch in text if ch in self.char2idx]

    def decode(self, indices: list[int], remove_duplicates=True, remove_blank=True) -> str:
        result = []
        prev = None
        for idx in indices:
            # collapse consecutive duplicates (raw CTC output holds characters across many timesteps)
            if remove_duplicates and idx == prev:
                prev = idx
                continue
            prev = idx
            # strip blank tokens
            if remove_blank and idx == self.blank_idx:
                continue
            if idx in self.idx2char:
                result.append(self.idx2char[idx])
        return "".join(result)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
class IAMLineDataset(Dataset):
    def __init__(self, split: str, vocab: Vocab, transform=None):
        print(f"Loading IAM-line [{split}] from HuggingFace")
        self.data = load_dataset("Teklia/IAM-line", split=split)
        self.vocab = vocab
        self.transform = transform
        print(f"  -> {len(self.data)} samples loaded.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item["image"]
        text = item["text"].lower()

        if self.transform:
            image = self.transform(image)  # PIL image -> (1, 32, W) tensor

        # dtype=long required specifically by nn.CTCLoss for target labels
        label = torch.tensor(self.vocab.encode(text), dtype=torch.long)
        # return raw text too so eval can compute CER/WER
        return image, label, text

In [ ]:
def get_dataloaders(vocab: Vocab, img_height: int = 32, batch_size: int = 32):
    train_ds = IAMLineDataset("train",      vocab, transform=get_transforms(img_height, augment=True))
    val_ds   = IAMLineDataset("validation", vocab, transform=get_transforms(img_height, augment=False))
    test_ds  = IAMLineDataset("test",       vocab, transform=get_transforms(img_height, augment=False))

    # shuffle training data each epoch, no need to shuffle val/test
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader

In [ ]:
import editdistance

def compute_cer(preds, text):
    total_dist = 0
    total_chars = 0
    for p, g in zip(preds, text):
        total_dist += editdistance.eval(p, g)
        total_chars += len(g)
    return total_dist / total_chars

In [ ]:
def compute_wer(preds, texts):
    total_dist = 0
    total_words = 0
    for p, g in zip(preds, texts):
        total_dist += editdistance.eval(p.split(), g.split())
        total_words += len(g.split())
    return total_dist / total_words

In [ ]:
def train_model(model, train_dl, opt, criterion, val_dl, epochs, device, vocab, best_cer=float('inf')):
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_idx, (images_tensor, labels_concat, label_lens, input_lens, texts) in enumerate(train_dl):
            images = images_tensor.to(device)
            labels_concat = labels_concat.to(device)
            label_lens = label_lens.to(device)
            input_lens = input_lens.to(device)
            opt.zero_grad()
            z = model(images)

            # get predicted probabilities for each timestep
            log_probs = torch.nn.functional.log_softmax(z.permute(1, 0, 2), dim=2)
            loss = criterion(log_probs, labels_concat, input_lens, label_lens)
            loss.backward()

            # stabilizes
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()
            train_loss += loss.item()

            # print a sample prediction every 50 batches
            if batch_idx % 50 == 0:
                with torch.no_grad():
                    sample_pred = vocab.decode(torch.argmax(z, dim=2)[0].tolist())
                print(f"[batch {batch_idx}] Sample pred: '{sample_pred}' | true: '{texts[0]}'")

        model.eval()
        val_loss = 0
        all_preds = []
        all_actual_texts = []
        with torch.no_grad():
            for images_tensor, labels_concat, label_lens, input_lens, texts in val_dl:
                images = images_tensor.to(device)
                labels_concat = labels_concat.to(device)
                label_lens = label_lens.to(device)
                input_lens = input_lens.to(device)
                z = model(images)

                log_probs = torch.nn.functional.log_softmax(z.permute(1, 0, 2), dim=2)
                loss = criterion(log_probs, labels_concat, input_lens, label_lens)
                val_loss += loss.item()

                preds = torch.argmax(z, dim=2)  # (B, W)
                for b in range(preds.size(0)):
                    indices = preds[b].tolist()
                    pred_text = vocab.decode(indices)
                    all_preds.append(pred_text)
                    all_actual_texts.append(texts[b])

        val_cer = compute_cer(all_preds, all_actual_texts)
        val_wer = compute_wer(all_preds, all_actual_texts)

        if val_cer < best_cer:
            best_cer = val_cer
            torch.save(model.state_dict(), checkpoint_path)
            torch.save(model.state_dict(), "best_model.pt")
            print(f"Saved new best model with cer: {best_cer}")

        print(f"Epoch {epoch+1}/{epochs} "
                f"| train loss: {train_loss/len(train_dl):.4f} "
                f"| val loss: {val_loss/len(val_dl):.4f}"
                f"| val cer: {val_cer}"
                f"| val wer: {val_wer}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = Vocab()
model = CRNN(vocab_size=vocab.size).to(device)
epochs = 20

train_dl, val_dl, test_dl = get_dataloaders(vocab, img_height=64, batch_size=32)

# CTC loss, zero_infinity prevents NaN losses
criterion = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
opt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [ ]:
train_model(model, train_dl, opt, criterion, val_dl, epochs, device, vocab)

In [ ]:
model.eval()
with torch.no_grad():
    images, labels_concat, label_lens, input_lens, texts = next(iter(val_dl))
    output = model(images.to(device))
    indices = torch.argmax(output, dim=2)

    for i in range(5):
        pred = vocab.decode(indices[i].tolist())
        print(f"pred:  '{pred}'")
        print(f"true:  '{texts[i]}'")
        print()

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    images, labels_concat, label_lens, input_lens, texts = next(iter(test_dl))
    output = model(images.to(device))
    indices = torch.argmax(output, dim=2)

    correct = []
    wrong = []

    for i in range(len(texts)):
        pred = vocab.decode(indices[i].tolist())
        true = texts[i]
        if pred == true and len(correct) < 3:
            correct.append((i, pred, true))
        elif pred != true and len(wrong) < 3:
            wrong.append((i, pred, true))
        if len(correct) == 3 and len(wrong) == 3:
            break

    for label, samples in [("✓ CORRECT", correct), ("✗ WRONG", wrong)]:
        print(f"\n{'='*50} {label} {'='*50}")
        for i, pred, true in samples:
            img = images[i].permute(1, 2, 0).cpu().numpy()
            img = (img * 0.5) + 0.5  # undo normalization

            plt.figure(figsize=(14, 2))
            plt.imshow(img, cmap='gray')
            plt.axis('off')
            # plt.title(f"pred: '{pred}'\ntrue: '{true}'", fontsize=10, loc='left')
            plt.tight_layout()
            plt.show()
            print(f"pred: '{pred}'")
            # print(f"true: '{true}'\n")

In [ ]:
model.load_state_dict(torch.load(checkpoint_path))

In [ ]:
all_preds = []
all_actual_texts = []
test_loss = 0
i = 0
with torch.no_grad():
    for images_tensor, labels_concat, label_lens, input_lens, texts in test_dl:
        images = images_tensor.to(device)
        labels_concat = labels_concat.to(device)
        label_lens = label_lens.to(device)
        input_lens = input_lens.to(device)
        z = model(images)


        log_probs = torch.nn.functional.log_softmax(z.permute(1, 0, 2), dim=2)
        loss = criterion(log_probs, labels_concat, input_lens, label_lens)
        test_loss += loss.item()

        preds = torch.argmax(z, dim=2)
        for b in range(preds.size(0)):
            indices = preds[b].tolist()
            pred_text = vocab.decode(indices)

            all_preds.append(pred_text)
            all_actual_texts.append(texts[b])
        print("batch tested", i)
        i+=1

    test_cer = compute_cer(all_preds, all_actual_texts)
    test_wer = compute_wer(all_preds, all_actual_texts)

Code to see how model performs on input image.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# load image
img = Image.open("varsha_ex.png")

# apply the same transforms used during training
transform = get_transforms(img_height=64, augment=False)
tensor = transform(img).unsqueeze(0).to(device) 

# run through model
model.eval()
with torch.no_grad():
    output = model(tensor)
    pred = vocab.decode(torch.argmax(output, dim=2)[0].tolist())

# show image and prediction
plt.figure(figsize=(14, 2))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.title(f"pred: '{pred}'", fontsize=12)
plt.show()
print(f"Predicted text: '{pred}'")